# Notebook 10: Baseline Neural Collaborative Filtering (NCF / NeuMF)

**FedSocial V2 — Institutional Recommender Systems & Federated Learning Research**  
**Subsystem**: Deep Neural Ranking Baselines  
**Architecture**: Generalized Matrix Factorization (GMF) + Multi-Layer Perceptron (MLP) Fusion

---

## 1. NeuMF Theoretical Architecture
NeuMF models non-linear user-item interactions by fusing generalized element-wise product with multi-layer perceptron representations:
$$\phi^{GMF} = \mathbf{p}_u^G \odot \mathbf{q}_i^G$$
$$\phi^{MLP} = a_L \left( \mathbf{W}_L^T \dots a_1(\mathbf{W}_1^T [\mathbf{p}_u^M, \mathbf{q}_i^M] + \mathbf{b}_1) \dots + \mathbf{b}_L \right)$$
$$\hat{y}_{ui} = \sigma \left( \mathbf{h}^T [\phi^{GMF}, \phi^{MLP}] \right)$$
Where $\mathbf{h}$ is the prediction layer weight vector and $\sigma$ is the sigmoid activation.


In [1]:
import torch
import torch.nn as nn

class NeuMF(nn.Module):
    def __init__(self, num_users, num_items, latent_dim=16, mlp_layers=[64, 32, 16]):
        super().__init__()
        # GMF embeddings
        self.user_embed_gmf = nn.Embedding(num_users, latent_dim)
        self.item_embed_gmf = nn.Embedding(num_items, latent_dim)
        # MLP embeddings
        self.user_embed_mlp = nn.Embedding(num_users, latent_dim)
        self.item_embed_mlp = nn.Embedding(num_items, latent_dim)
        # MLP Layers
        layers = []
        in_dim = latent_dim * 2
        for out_dim in mlp_layers:
            layers.append(nn.Linear(in_dim, out_dim))
            layers.append(nn.ReLU())
            in_dim = out_dim
        self.mlp = nn.Sequential(*layers)
        # Final prediction layer
        self.predict_layer = nn.Linear(latent_dim + mlp_layers[-1], 1)

    def forward(self, u, i):
        gmf = self.user_embed_gmf(u) * self.item_embed_gmf(i)
        mlp_in = torch.cat([self.user_embed_mlp(u), self.item_embed_mlp(i)], dim=-1)
        mlp_out = self.mlp(mlp_in)
        out = self.predict_layer(torch.cat([gmf, mlp_out], dim=-1))
        return torch.sigmoid(out).squeeze()

model = NeuMF(num_users=100, num_items=5444)
print(f'NeuMF Total Trainable Parameters: {sum(p.numel() for p in model.parameters()):,}')


NeuMF Total Trainable Parameters: 352,241
